# Milestone 2: multi-object detection (Google Colab)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
DATA_LINK = ''

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/IE7615_Milestone2'
except Exception as e:
    print('Google Drive not mounted, outputs stay in /content/outputs:', e)
    DRIVE = '/content/outputs'
RAW = f'{DRIVE}/raw'
REPO = 'https://github.com/anshuman-dataverse/discriminative-deep-learning-project.git'
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
%cd /content
!test -d project || git clone -q $REPO project
!cd project && git pull -q
!pip -q install ultralytics gdown
if DATA_LINK:
    !test -f /content/raw.zip || gdown --fuzzy "$DATA_LINK" -O /content/raw.zip
    !mkdir -p /content/raw && unzip -qn /content/raw.zip -d /content/raw
else:
    !rsync -a "$RAW/" /content/raw/
!ls /content/raw | head -3; ls /content/raw | wc -l

## 1. Clean and split the single-object photos

In [ ]:
%cd /content/project/milestone2/scripts
!mkdir -p ../runs
!python 00_prepare_singles.py /content/raw 2>&1 | tee ../runs/prepare_singles.log

## 2. Bounding boxes from Grounding DINO and OWLv2

In [ ]:
import os, shutil
cache = f'{DRIVE}/boxes.csv'
if os.path.exists(cache):
    shutil.copy(cache, '../data/singles/boxes.csv')
    print('restored boxes.csv from Drive')
else:
    !python 01_label_boxes.py 2>&1 | tee ../runs/label_boxes.log
    shutil.copy('../data/singles/boxes.csv', cache)

In [ ]:
from IPython.display import Image, display
display(Image('../screenshots/label_check_kept.jpg', width=1000))
display(Image('../screenshots/label_check_dropped.jpg', width=1000))

## 3. Grid images (2x2 to 5x5)

In [ ]:
!python 02_make_grids.py 2>&1 | tee ../runs/make_grids.log
display(Image('../screenshots/multi_object_samples.jpg', width=1200))

## 4. Train YOLOv8s (imgsz 1120, batch 8, checkpoints on Drive)

In [ ]:
!python 03_train_yolo.py --project "$DRIVE/runs" --imgsz 1120 --batch 8 --epochs 60 2>&1 | tee -a ../runs/train_yolov8s.log

## 5. Test-set evaluation and example detections

In [ ]:
!python 04_evaluate_yolo.py --imgsz 1120 2>&1 | tee ../runs/evaluate.log
!cat ../results/test_metrics.json | head -20
display(Image('../screenshots/example_detections.jpg', width=1200))

In [ ]:
!python 05_detect.py ../data/multi/images/test --limit 10 2>&1 | tee ../runs/detect.log

## 6. Save outputs to Drive

In [ ]:
!rsync -a "$DRIVE/runs/yolov8s/" ../runs/yolov8s/ --exclude weights
!cd .. && zip -qr "$DRIVE/milestone2_outputs.zip" results screenshots models runs -x 'runs/detect/*' 'runs/*/weights/*'
!ls -lh "$DRIVE"
if DRIVE == '/content/outputs':
    from google.colab import files
    files.download(f'{DRIVE}/milestone2_outputs.zip')